In [1]:
import pandas as pd
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

In [2]:
df = pd.read_csv(r'D:\Python\我的AI作品集\專案1_數學函數辨識\data\v1\function_dataset_cleaned.csv')

In [3]:
# 定義特徵與目標變數
X = df.drop(columns=['label'])
y = df['label']

## 將數據分割為訓練集與測試集

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

In [5]:
print(f"訓練集類別:\n{y_train.value_counts()}\n")

訓練集類別:
label
cubic          400
cosine         400
exponential    400
quadratic      400
reciprocal     400
sine           400
logarithmic    400
linear         400
Name: count, dtype: int64



In [6]:
print(f"測試集類別:\n{y_test.value_counts()}\n")

測試集類別:
label
quadratic      100
logarithmic    100
reciprocal     100
sine           100
cosine         100
exponential    100
cubic          100
linear         100
Name: count, dtype: int64



## 蒐集模型, 建立pipeline

In [7]:
models = {
    "LogisticRegression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    ),
    "DecisionTree": DecisionTreeClassifier(random_state=10),

    "RandomForest": RandomForestClassifier(random_state=10),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=3)
    ),
    "SVM": make_pipeline(
        StandardScaler(),
        SVC()
    )
}

In [8]:
for name, model in models.items():
    start_time = time.perf_counter()        # 開始計時
    model.fit(X_train, y_train)
    end_time = time.perf_counter()          # 結束計時
    elapsed_time = end_time - start_time    # 計算經過時間
    print(f"{name} 模型訓練完成, 耗時: {elapsed_time:.3f}秒\n")

    # 儲存到joblib
    joblib.dump(model, rf'D:\Python\我的AI作品集\專案1_數學函數辨識\models\v1\{name}_baseline.joblib')

LogisticRegression 模型訓練完成, 耗時: 0.520秒

DecisionTree 模型訓練完成, 耗時: 0.250秒

RandomForest 模型訓練完成, 耗時: 2.853秒

KNN 模型訓練完成, 耗時: 0.012秒

SVM 模型訓練完成, 耗時: 0.200秒



## 預測模型

In [9]:
from sklearn.metrics import *

In [10]:
results = []
trained_models = {}

In [11]:
# 預測模型
for name, model in models.items():
    model = joblib.load(rf'D:\Python\我的AI作品集\專案1_數學函數辨識\models\v1\{name}_baseline.joblib')
    trained_models[name] = model        # 儲存模型物件

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"{name} 模型計算準確度: {acc:.4f}")
    print(f"{name} 模型的混淆矩陣:\n{confusion_matrix(y_test, y_pred)}\n")
    print(f"{name} 模型的分類報告:\n{classification_report(y_test, y_pred)}\n")
    results.append({'Model': name, 
                    'Accuracy': acc, 
                    'Precision': pre, 
                    'Recall': recall, 
                    'F1': f1})

LogisticRegression 模型計算準確度: 0.4550
LogisticRegression 模型的混淆矩陣:
[[90  2  3  2  0  0  3  0]
 [ 2 15 15 19 14 15 13  7]
 [ 0  0 79  2  3  5 11  0]
 [ 0 15 16  9 19 20 16  5]
 [ 5  4 21 12 20 12 26  0]
 [ 0 19 19 16 10 19 17  0]
 [ 0  2 27  6 16  9 40  0]
 [ 1  2  2  1  1  1  0 92]]

LogisticRegression 模型的分類報告:
              precision    recall  f1-score   support

      cosine       0.92      0.90      0.91       100
       cubic       0.25      0.15      0.19       100
 exponential       0.43      0.79      0.56       100
      linear       0.13      0.09      0.11       100
 logarithmic       0.24      0.20      0.22       100
   quadratic       0.23      0.19      0.21       100
  reciprocal       0.32      0.40      0.35       100
        sine       0.88      0.92      0.90       100

    accuracy                           0.46       800
   macro avg       0.43      0.46      0.43       800
weighted avg       0.43      0.46      0.43       800


DecisionTree 模型計算準確度: 0.7100
DecisionTr

In [12]:
results_df = pd.DataFrame(results)

In [13]:
# 儲存成網頁
results_df.to_html(r'D:\Python\我的AI作品集\專案1_數學函數辨識\data\v1\Model_Comparison.html')

## 模型比較

In [14]:
print("模型效能比較:")
print(results_df.to_string(index=False, formatters={'Accuracy': '{:.3f}'.format, 
                                                    'Precision': '{:.3f}'.format, 
                                                    'Recall': '{:.3f}'.format, 
                                                    'F1': '{:.3f}'.format}))

模型效能比較:
             Model Accuracy Precision Recall    F1
LogisticRegression    0.455     0.427  0.455 0.431
      DecisionTree    0.710     0.716  0.710 0.711
      RandomForest    0.821     0.824  0.821 0.821
               KNN    0.810     0.816  0.810 0.810
               SVM    0.760     0.787  0.760 0.763


### 找出 F1 最高的模型

In [15]:
# 找出 F1 最高的模型
best_result = results_df.loc[results_df['F1'].idxmax()]
best_model_name = best_result['Model']
best_model = trained_models[best_model_name]
print(f"\n最佳模型:\n{best_result}")


最佳模型:
Model        RandomForest
Accuracy          0.82125
Precision        0.824403
Recall            0.82125
F1               0.821004
Name: 2, dtype: object


In [16]:
joblib.dump(best_model, r'D:\Python\我的AI作品集\專案1_數學函數辨識\models\v1\best_model.joblib')

['D:\\Python\\我的AI作品集\\專案1_數學函數辨識\\models\\v1\\best_model.joblib']